# Factual-consistency evaluation: AlignScore-base, SummaC-Conv, BERTScore and BARTScore

This notebook compares the four pretrained methods on the summarization portions of RAGTruth and HaluEval, and on FaithBench. It scores identical selected examples across methods, with `1` indicating a dataset-annotated or benchmark-designated hallucination and `0` otherwise.

**Works unchanged in Google Colab:** use a GPU runtime, configure email above if desired, then run cells in order. In Colab, results still go to timestamped folders on mounted Google Drive. **On a Google Cloud VM**, execute from the notebook's directory; outputs and cloned dependencies live alongside the notebook, and the latest interrupted VM run resumes automatically.

This is an evaluation on new datasets, not an exact reproduction of the original papers' benchmark results. Label policies differ across datasets (see below).


In [ ]:
# CONFIGURATION — set these values before running the notebook.
# Never publish a notebook containing a real Gmail app password.
EMAIL_FROM = "germanypicturedump@gmail.com"  # Gmail address sending progress reports
EMAIL_TO = [
    "elimyoseph30@gmail.com",
    "theryanchance@gmail.com",
]  # Recipients: ["one@gmail.com", "two@gmail.com"]. Empty defaults to EMAIL_FROM
EMAIL_APP_PASSWORD = "enuh ckek gnat adxs"  # 16-character Google app password, OR set env GMAIL_APP_PASSWORD
EMAIL_EVERY_MINUTES = 15  # Progress email interval
EMAIL_MAX_ATTACHMENT_MB = 18  # Stay below Gmail's encoded attachment limit
REPORT_TIMEZONE = "America/Toronto"

# VM-only options; the existing Colab paths/Drive behavior stay unchanged.
REQUIRE_GPU_ON_VM = True  # Stop instead of unintentionally scoring everything on CPU
RESUME_LATEST_ON_VM = True  # Reuse the newest run_* folder in the notebook's directory
RESUME_RUN_DIR = ""  # Optional explicit previous run folder (also works in Colab)

# The notebook kernel normally starts in the notebook's folder. On a VM, cd there
# before running with nbconvert. Colab keeps its original /content/ directory.
import os, sys, json, time, threading, smtplib, ssl, subprocess, traceback, html
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from contextlib import contextmanager
from email.message import EmailMessage

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_DIR = Path("/content") if IN_COLAB else Path.cwd().resolve()
EVAL_SCRIPT = WORK_DIR / "evaluate_comparison.py"
BARTSCORE_DIR = WORK_DIR / "BARTScore"
EMAIL_APP_PASSWORD = EMAIL_APP_PASSWORD or os.getenv("GMAIL_APP_PASSWORD", "")
EMAIL_TO = EMAIL_TO or EMAIL_FROM
print("Environment:", "Google Colab" if IN_COLAB else "Local / Google Cloud VM")
print("Working directory:", WORK_DIR)


class EmailReporter:
    def __init__(self, run_dir):
        self.run_dir = Path(run_dir)
        self.progress_path = Path(
            os.environ.get(
                "FACTUALITY_PROGRESS_FILE", str(self.run_dir / "progress.json")
            )
        )
        self.evaluator_pid = None
        self.phase = "Preparing environment"
        self.started = time.time()
        self.stop_event = threading.Event()
        self.thread = None
        # Accept a list of addresses or the original comma/semicolon-separated string.
        addresses = [EMAIL_TO] if isinstance(EMAIL_TO, str) else EMAIL_TO
        self.recipients = list(
            dict.fromkeys(
                address.strip()
                for item in addresses
                for address in item.replace(";", ",").split(",")
                if address.strip()
            )
        )
        self.enabled = bool(
            EMAIL_FROM.strip() and self.recipients and EMAIL_APP_PASSWORD.strip()
        )
        self.tz = ZoneInfo(REPORT_TIMEZONE)

    def set_phase(self, phase):
        self.phase = phase

    def progress(self):
        try:
            return json.loads(self.progress_path.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return {}

    @staticmethod
    def fmt_duration(seconds):
        if seconds is None:
            return "Calculating..."
        seconds = max(0, int(seconds))
        hours, remainder = divmod(seconds, 3600)
        minutes, _ = divmod(remainder, 60)
        return f"{hours}h {minutes:02d}m" if hours else f"{minutes}m"

    @staticmethod
    def bar(percent):
        n = max(0.0, min(100.0, float(percent)))
        return (
            f'<div style="background:#e8edf5;border-radius:20px;height:8px;overflow:hidden">'
            f'<div style="background:#4765dc;width:{n:.3f}%;height:8px"></div></div>'
        )

    def resources(self):
        lines = []
        try:
            import psutil

            vm = psutil.virtual_memory()
            lines.append(f"CPU: {psutil.cpu_percent(interval=None):.0f}% (whole VM)")
            lines.append(
                f"RAM: {vm.used / 2**30:.1f} / {vm.total / 2**30:.1f} GiB ({vm.percent:.0f}%)"
            )
            lines.append(
                f"Notebook RAM: {psutil.Process().memory_info().rss / 2**30:.2f} GiB"
            )
            if self.evaluator_pid and psutil.pid_exists(self.evaluator_pid):
                worker = psutil.Process(self.evaluator_pid)
                lines.append(
                    f"Evaluation process RAM: {worker.memory_info().rss / 2**30:.2f} GiB"
                )
            disk = psutil.disk_usage(str(self.run_dir))
            lines.append(
                f"Disk: {disk.used / 2**30:.1f} / {disk.total / 2**30:.1f} GiB ({disk.percent:.0f}%)"
            )
        except Exception:
            lines.append("CPU/RAM: install psutil for detailed usage")
        try:
            proc = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=name,utilization.gpu,memory.used,memory.total",
                    "--format=csv,noheader,nounits",
                ],
                capture_output=True,
                text=True,
                timeout=5,
                check=True,
            )
            for gpu in proc.stdout.strip().splitlines():
                if gpu.strip():
                    lines.append("GPU (name, %, VRAM used/total MiB): " + gpu.strip())
        except (OSError, subprocess.SubprocessError):
            lines.append("GPU: nvidia-smi unavailable")
        return lines

    def build(self, status, details="", attachment_note=""):
        p = self.progress()
        pct = 100.0 if status == "COMPLETED" else float(p.get("percent", 0))
        remaining = 0 if status == "COMPLETED" else p.get("remaining_seconds")
        now = datetime.now(self.tz)
        eta = (
            (now + timedelta(seconds=remaining)).strftime("%b %d %H:%M %Z")
            if remaining is not None
            else "Calculating..."
        )
        subject = (
            f"{pct:.2f}% | ETA {eta} | {self.fmt_duration(remaining)} remaining"
            f" — Factuality evaluation: {status}"
        )
        stage = p.get("phase", self.phase) if p else self.phase
        method = p.get("current_method", "Not started")
        benchmark = p.get("current_dataset", "Not started")
        task = p.get("current_split", "Not started")
        example = p.get("current_id", "—")
        overall_count = f'{p.get("completed", 0):,} / {p.get("total", "? ")} scores'
        elapsed = self.fmt_duration(time.time() - self.started)
        rows = []
        for ds, vals in p.get("benchmarks", {}).items():
            done, total = vals.get("completed", 0), vals.get("total", 0)
            value = 100 * done / total if total else 0
            rows.append(
                f'<tr><td style="padding:9px;border-bottom:1px solid #e5e9f0">{html.escape(ds)}</td>'
                f'<td style="padding:9px;border-bottom:1px solid #e5e9f0">{done:,}/{total:,}</td>'
                f'<td style="padding:9px;border-bottom:1px solid #e5e9f0">{value:.2f}%{self.bar(value)}</td></tr>'
            )
        methods = []
        for name, vals in p.get("methods", {}).items():
            done, total = vals.get("completed", 0), vals.get("total", 0)
            value = 100 * done / total if total else 0
            methods.append(
                f'<tr><td style="padding:9px;border-bottom:1px solid #e5e9f0">{html.escape(name)}</td>'
                f'<td style="padding:9px;border-bottom:1px solid #e5e9f0">{done:,}/{total:,}</td>'
                f'<td style="padding:9px;border-bottom:1px solid #e5e9f0">{value:.2f}%{self.bar(value)}</td></tr>'
            )
        task_counts = p.get("current_task", {})
        current_benchmark = p.get("current_benchmark", {})
        current_method = p.get("current_method_progress", {})
        task_value = task_counts.get("percent", 0)
        benchmark_value = current_benchmark.get("percent", 0)
        method_value = current_method.get("percent", 0)

        def card(label, value, desc):
            return (
                f'<td style="width:33%;vertical-align:top;padding:10px;background:#f4f6fa;border:6px solid #fff">'
                f'<div style="font-size:12px;color:#64748b">{label}</div>'
                f'<div style="font-size:22px;font-weight:700;margin:6px 0">{value:.2f}%</div>'
                f'{self.bar(value)}<div style="font-size:11px;color:#64748b;margin-top:8px">{html.escape(desc)}</div></td>'
            )

        def counts(v):
            return f'{v.get("completed", 0):,} / {v.get("total", 0):,} scores'

        details_block = (
            (
                f'<div style="white-space:pre-wrap;background:#fff4f3;border:1px solid #ffd1cd;'
                f'border-radius:8px;padding:14px;color:#922e24">{html.escape(details[-6500:])}</div>'
            )
            if details
            else ""
        )
        note_block = (
            f'<p style="color:#944c19">{html.escape(attachment_note)}</p>'
            if attachment_note
            else ""
        )
        body = f"""<!doctype html><html><body style="margin:0;background:#f3f5fa;font-family:Arial,Helvetica,sans-serif;color:#1f2937">
        <div style="max-width:730px;margin:auto;padding:24px 12px"><div style="background:#fff;border-radius:14px;overflow:hidden">
          <div style="padding:26px;background:#152343;color:#fff">
            <div style="font-size:11px;font-weight:bold;letter-spacing:2px;color:#b9c9ef">RESEARCH MONITOR</div>
            <div style="font-size:23px;margin:12px 0 3px">Factuality evaluation</div>
            <div style="font-size:14px;color:#d9e3ff">{html.escape(status)} · {now.strftime('%b %d, %Y · %H:%M %Z')}</div>
            <div style="font-size:38px;font-weight:700;margin-top:20px">{pct:.2f}%</div>
            {self.bar(pct)}
            <div style="font-size:13px;margin-top:12px">{html.escape(overall_count)} &nbsp; | &nbsp; ETA: {html.escape(eta)} &nbsp; | &nbsp; {html.escape(self.fmt_duration(remaining))} remaining</div>
          </div>
          <div style="padding:24px">
            <div style="font-size:13px;color:#64748b">CURRENT STAGE</div>
            <div style="font-size:17px;font-weight:bold;margin:5px 0 8px">{html.escape(stage)}</div>
            <p style="font-size:13px;line-height:1.8">Method: <b>{html.escape(method)}</b><br>
               Benchmark: <b>{html.escape(benchmark)}</b> · Task: <b>{html.escape(task)}</b><br>
               Last example: <code>{html.escape(str(example)[:80])}</code></p>
            <table style="width:100%;border-collapse:collapse"><tr>
              {card('METHOD', method_value, counts(current_method))}
              {card('BENCHMARK IN METHOD', benchmark_value, counts(current_benchmark))}
              {card('VALIDATION / TEST TASK', task_value, counts(task_counts))}
            </tr></table>
            <h3 style="font-size:15px;margin:26px 0 8px">Benchmarks · across all methods</h3>
            <table style="border-collapse:collapse;width:100%;font-size:12px"><tbody>{''.join(rows) or '<tr><td>Preparing datasets...</td></tr>'}</tbody></table>
            <h3 style="font-size:15px;margin:26px 0 8px">Method progress</h3>
            <table style="border-collapse:collapse;width:100%;font-size:12px"><tbody>{''.join(methods) or '<tr><td>Preparing models...</td></tr>'}</tbody></table>
            <h3 style="font-size:15px;margin:26px 0 8px">System resources</h3>
            <div style="font-family:monospace;font-size:12px;line-height:1.9;background:#f4f6fa;padding:13px;border-radius:8px">{''.join(html.escape(x)+'<br>' for x in self.resources())}</div>
            <p style="font-size:12px;color:#64748b">Session runtime: {html.escape(elapsed)}. ETA is an estimate from observed scoring throughput and may change between models.</p>
            {details_block}{note_block}
          </div></div>
          <p style="text-align:center;color:#94a3b8;font-size:11px">Automatic research progress report · {html.escape(str(self.run_dir))}</p>
        </div></body></html>"""
        plain = (
            f"{status}: {pct:.2f}% complete. ETA {eta}; {self.fmt_duration(remaining)} remaining.\n"
            f"Stage: {stage} | {method} | {benchmark} | {task} | example {example}\n"
            + "\n".join(self.resources())
            + "\n"
            + details
            + "\n"
            + attachment_note
        )
        return subject, plain, body

    def send(self, status="PROGRESS", details="", attachment=None):
        if not self.enabled:
            return
        note = ""
        attach = Path(attachment) if attachment else None
        if (
            attach
            and attach.exists()
            and attach.stat().st_size > EMAIL_MAX_ATTACHMENT_MB * 1024 * 1024
        ):
            note = f"Results ZIP is too large to attach ({attach.stat().st_size / 2**20:.1f} MiB). Saved at: {attach}"
            attach = None
        for attempt in (0, 1):
            subject, plain, rich = self.build(status, details, note)
            msg = EmailMessage()
            msg["From"] = EMAIL_FROM.strip()
            msg["To"] = ", ".join(self.recipients)
            msg["Subject"] = subject
            msg.set_content(plain)
            msg.add_alternative(rich, subtype="html")
            if attach is not None:
                try:
                    msg.add_attachment(
                        attach.read_bytes(),
                        maintype="application",
                        subtype="zip",
                        filename=attach.name,
                    )
                except Exception as exc:
                    print(
                        f"Could not attach results ZIP ({exc}); sending completion notice without it."
                    )
                    note = (
                        "Results ZIP could not be attached. It remains saved at: "
                        + str(attach)
                    )
                    attach = None
                    continue
            try:
                with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as server:
                    server.ehlo()
                    server.starttls(context=ssl.create_default_context())
                    server.ehlo()
                    server.login(
                        EMAIL_FROM.strip(), EMAIL_APP_PASSWORD.replace(" ", "")
                    )
                    server.send_message(msg)
                print(f"Email sent: {status}")
                return
            except Exception as exc:
                if attach is not None:
                    print(
                        f"Attachment email failed ({type(exc).__name__}); retrying without ZIP."
                    )
                    note = f"Results ZIP was saved but could not be emailed: {attach}"
                    attach = None
                else:
                    print(
                        f"Email notification failed ({type(exc).__name__}: {exc}). Results are still on disk."
                    )
                    return

    def start(self):
        if not self.enabled:
            print(
                "Email reports disabled: fill EMAIL_FROM and EMAIL_APP_PASSWORD in CONFIGURATION."
            )
            return

        def loop():
            interval = max(0.1, float(EMAIL_EVERY_MINUTES)) * 60
            while not self.stop_event.wait(interval):
                self.send("PROGRESS")

        self.thread = threading.Thread(
            target=loop, daemon=True, name="factuality-email-reporter"
        )
        self.thread.start()
        self.send("STARTED")

    def stop(self):
        self.stop_event.set()
        if self.thread and threading.current_thread() != self.thread:
            self.thread.join(timeout=1)


@contextmanager
def report_errors(stage):
    REPORTER.set_phase(stage)
    try:
        yield
    except BaseException:
        REPORTER.send("ERROR", traceback.format_exc())
        REPORTER.stop()
        raise

## Results folder and automatic environment selection
In Colab, mount Drive and use the original `MyDrive/EECS_6412_factuality_results1000` base path with a new timestamped run folder (same behavior as before). In a VM/local Jupyter session, use a timestamped `run_*` folder next to the notebook. Set `RESUME_RUN_DIR` in the configuration to resume a specific run; on a VM `RESUME_LATEST_ON_VM=True` automatically reuses the latest `run_*` folder. Every completed batch of 10 predictions is checkpointed and automatically resumed. The final CSVs contain every example exactly once.


In [ ]:
from datetime import datetime

if IN_COLAB:
    from google.colab import drive

    try:
        drive.mount("/content/drive")
    except BaseException:
        EmailReporter(WORK_DIR).send("ERROR", traceback.format_exc())
        raise
    DRIVE_BASE = Path("/content/drive/MyDrive/EECS_6412_factuality_results1000")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    DEFAULT_RUN_BASE = DRIVE_BASE
else:
    DEFAULT_RUN_BASE = WORK_DIR

if RESUME_RUN_DIR:
    RUN_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    if not RUN_DIR.is_dir():
        raise FileNotFoundError(f"Resume folder does not exist: {RUN_DIR}")
elif not IN_COLAB and RESUME_LATEST_ON_VM:
    runs = sorted(
        (
            p
            for p in DEFAULT_RUN_BASE.glob("run_*")
            if p.is_dir()
            and not (
                (p / "metrics.csv").exists() and (p.parent / (p.name + ".zip")).exists()
            )
        ),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    RUN_DIR = (
        runs[0]
        if runs
        else DEFAULT_RUN_BASE / datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
    )
else:
    RUN_DIR = DEFAULT_RUN_BASE / datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ["FACTUALITY_RESULTS_DIR"] = str(RUN_DIR)
os.environ["FACTUALITY_WORK_DIR"] = str(WORK_DIR)
os.environ["FACTUALITY_BARTSCORE_DIR"] = str(BARTSCORE_DIR)
os.environ["FACTUALITY_REQUIRE_GPU"] = (
    "1" if (not IN_COLAB and REQUIRE_GPU_ON_VM) else "0"
)
# Keep per-example progress writes on Colab's fast local disk, not mounted Drive.
# The final copy is included in the results ZIP. VMs use the run directory directly.
LIVE_PROGRESS_FILE = (
    WORK_DIR / ("factuality_progress_" + RUN_DIR.name + ".json")
    if IN_COLAB
    else RUN_DIR / "progress.json"
)
if IN_COLAB:
    LIVE_PROGRESS_FILE.unlink(missing_ok=True)
os.environ["FACTUALITY_PROGRESS_FILE"] = str(LIVE_PROGRESS_FILE)
print("Results folder:", RUN_DIR)
REPORTER = EmailReporter(RUN_DIR)
REPORTER.start()

## 1. Setup
Install the original pinned versions in an isolated Python 3.10 environment. In Colab, the environment remains `/content/alignscore-env` and everything else remains in `/content/`. On a VM the environment and cloned code are created in the notebook directory. A GPU is strongly recommended; by default a VM without a working CUDA GPU fails early rather than unexpectedly processing the entire corpus on CPU. Set `REQUIRE_GPU_ON_VM=False` to allow a CPU VM.


In [ ]:
with report_errors("Installing Python environment"):
    import subprocess, sys

    def run(command):
        subprocess.run(command, check=True)

    run([sys.executable, "-m", "pip", "install", "-q", "uv", "psutil"])
    ENV = WORK_DIR / "alignscore-env"
    PYTHON = str(ENV / "bin/python")
    if not ENV.exists():
        run(["uv", "venv", "--python", "3.10", str(ENV)])
    run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            PYTHON,
            "torch==1.13.1",
            "numpy==1.24.4",
            "transformers==4.30.2",
            "pytorch-lightning==1.9.5",
            "torchmetrics==0.11.4",
            "spacy==3.7.5",
            "nltk==3.8.1",
            "scikit-learn==1.3.2",
            "pandas==2.0.3",
            "huggingface-hub==0.25.2",
            "protobuf==3.20.0",
            "tensorboard==2.14.0",
            "datasets==2.14.7",
            "pyarrow==14.0.2",
            "jsonlines==2.0.0",
            "tqdm",
            "setuptools==65.5.0",
            "sentencepiece",
        ]
    )
    run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            PYTHON,
            "--no-deps",
            "git+https://github.com/yuh-zha/AlignScore.git",
            "summac==0.0.4",
        ]
    )
    run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            PYTHON,
            "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.0/en_core_web_sm-3.7.0-py3-none-any.whl",
        ]
    )
    run([PYTHON, "-c", "import nltk; nltk.download('punkt', quiet=True)"])
    print("Environment ready:", PYTHON)

In [ ]:
with report_errors("Installing BERTScore and downloading BARTScore"):
    subprocess.run(
        ["uv", "pip", "install", "--python", PYTHON, "bert-score==0.3.13"], check=True
    )
    if not BARTSCORE_DIR.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/neulab/BARTScore.git",
                str(BARTSCORE_DIR),
            ],
            check=True,
        )
    print("BERTScore installed. BARTScore repository:", BARTSCORE_DIR)

In [ ]:
with report_errors("Checking model imports"):
    import subprocess

    check_code = """
try:
    import sentencepiece, transformers, torch, pytorch_lightning, torchmetrics
    import spacy, nltk, sklearn, pandas, datasets, pyarrow, jsonlines, tqdm
    import alignscore, summac, bert_score, sys, os
    sys.path.insert(0, os.environ['FACTUALITY_BARTSCORE_DIR'])
    import bart_score
    print('SUCCESS: All imports loaded inside the virtual environment!')
except ImportError as e:
    print(f'FAILURE: Missing dependency: {e}')
    sys.exit(1)
"""
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    res = subprocess.run(
        [PYTHON, "-c", check_code], capture_output=True, text=True, env=env
    )
    print(res.stdout)
    if res.returncode != 0:
        raise RuntimeError("Imports check failed:\n" + res.stderr[-5000:])

## 2. Download the datasets
Each repository's exact commit is recorded. Rerunning in an existing Colab session reuses the clone. For the final report record the commits and environment file from the results folder.

In [ ]:
with report_errors("Cloning and verifying benchmark datasets"):
    REPOS = {
        "RAGTruth": "https://github.com/ParticleMedia/RAGTruth.git",
        "HaluEval": "https://github.com/RUCAIBox/HaluEval.git",
        "FaithBench": "https://github.com/vectara/FaithBench.git",
    }
    for name, url in REPOS.items():
        path = WORK_DIR / name
        if not path.exists():
            run(["git", "clone", "--depth", "1", url, str(path)])
        commit = subprocess.check_output(
            ["git", "-C", str(path), "rev-parse", "HEAD"], text=True
        ).strip()
        print(name, commit)

## 3. Configure and run the comparison

- **RAGTruth:** summarization responses of good quality; any annotated span is positive, including `implicit_true`. Official test membership is preserved; overlapping training source IDs are excluded from validation.
- **HaluEval:** benchmark-designated right and hallucinated summary pairs are grouped by document and split without source leakage.
- **FaithBench:** `Questionable` or `Unwanted` (including subtypes) makes a summary positive. Split by exact source document.

Each method's threshold is selected on the corresponding dataset's validation examples (maximizing hallucination F1); test examples do not set thresholds. AUROC and average precision use continuous scores. `N_VALIDATION=None` and `N_TEST=None` evaluate the entire eligible pools.

Predictions are checkpointed in **atomic batches of 10** to `RUN_DIR/checkpoints/`; an interrupted run can resume without rescoring saved examples. The evaluation updates a small live progress JSON after each scored example for granular global, per-method, per-dataset, and validation/test progress. The notebook's background reporter emails status every `EMAIL_EVERY_MINUTES`, including ETA and system resources.


In [ ]:
%%writefile {EVAL_SCRIPT}
import json, random, time, hashlib, subprocess, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, confusion_matrix, f1_score
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm


# for reproduction 
SEED = 42
N_VALIDATION = None  # None = full validation pool: ATTENTION BESTIE change this value to use the full vals
N_TEST = None        # None = full test pool ATTENTION BESTIE to test it change this value 
OUT = Path(os.environ['FACTUALITY_RESULTS_DIR'])
ROOT = Path(os.environ.get('FACTUALITY_WORK_DIR', '/content'))
BARTSCORE_DIR = Path(os.environ.get('FACTUALITY_BARTSCORE_DIR', str(ROOT / 'BARTScore')))
METHODS = ['AlignScore-base', 'SummaC-Conv', 'BERTScore', 'BARTScore']
PROGRESS_FILE = Path(os.environ.get('FACTUALITY_PROGRESS_FILE', str(OUT / 'progress.json'))) 
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)

def read_jsonl(path):
    with open(path, encoding='utf8') as f:
        return [json.loads(line) for line in f if line.strip()]

def source_hash(text):
    return hashlib.sha256(text.encode('utf8')).hexdigest()

def row(dataset, ident, group, split, context, response, gold, extra=''):
    return dict(dataset=dataset, id=str(ident), group=str(group), split=split,
                context=context, response=response, gold=int(gold), annotation_details=extra)

def ragtruth(root):
    sources = {str(x['source_id']):x for x in read_jsonl(root/'dataset/source_info.jsonl')}
    result=[]
    for x in read_jsonl(root/'dataset/response.jsonl'):
        s=sources[str(x['source_id'])]
        if s['task_type'] != 'Summary' or x.get('quality') != 'good': continue
        context=s['source_info']; response=x['response']
        if not isinstance(context,str) or not context.strip() or not response.strip(): continue
        result.append(row('RAGTruth', x['id'], x['source_id'], x['split'], context, response,
                          bool(x['labels']), json.dumps(x['labels'], ensure_ascii=False)))
    df=pd.DataFrame(result)
    # Keep official test untouched, remove any overlapping train sources.
    overlap=set(df.loc[df.split=='train','group']) & set(df.loc[df.split=='test','group'])
    print('RAGTruth overlapping source IDs excluded from training:', len(overlap))
    train=df[(df.split=='train') & ~df.group.isin(overlap)].copy()
    test=df[df.split=='test'].copy()
    split=GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    _, ix=next(split.split(train, groups=train.group))
    val=train.iloc[ix].copy(); val['split']='validation'
    return pd.concat([val,test],ignore_index=True)

def halueval(root):
    path=root/'data/summarization_data.json'
    result=[]
    for i,x in enumerate(read_jsonl(path)):
        context=x['document']
        if not context.strip(): continue
        group=source_hash(context)
        for key,label in [('right_summary',0),('hallucinated_summary',1)]:
            if x[key].strip(): result.append(row('HaluEval',f'{i}-{key}',group,'',context,x[key],label,key))
    return grouped_split(pd.DataFrame(result))

def faithbench(root):
    result=[]
    files=sorted((root/'data_for_release').glob('batch_*.json'))
    if not files: raise RuntimeError('FaithBench batch files not found')
    for path in files:
        for x in json.loads(path.read_text(encoding='utf8'))['samples']:
            context=x['source']; response=x['summary']
            if not context.strip() or not response.strip(): continue
            labels=sorted({label for a in x.get('annotations',[]) for label in a.get('label',[])})
            unknown=set(labels)-{'Benign','Questionable','Unwanted','Unwanted.Intrinsic','Unwanted.Extrinsic'}
            if unknown: raise ValueError(f'Unexpected FaithBench labels: {unknown}')
            positive=any(s == 'Questionable' or s.startswith('Unwanted') for s in labels)
            result.append(row('FaithBench',f'{path.stem}-{x["sample_id"]}',source_hash(context), '',context,response,positive,json.dumps(labels)))
    return grouped_split(pd.DataFrame(result))

def grouped_split(df):
    splitter=GroupShuffleSplit(n_splits=1,test_size=0.8,random_state=SEED)
    vi,ti=next(splitter.split(df,groups=df.group))
    val=df.iloc[vi].copy(); val['split']='validation'
    test=df.iloc[ti].copy(); test['split']='test'
    return pd.concat([val,test],ignore_index=True)

def sample(df,n):
    if n is None or n >= len(df): return df.copy().reset_index(drop=True)
    return df.sample(n=n,random_state=SEED).reset_index(drop=True)

def prepare():
    data=pd.concat([ragtruth(ROOT / 'RAGTruth'),halueval(ROOT / 'HaluEval'),
                    faithbench(ROOT / 'FaithBench')],ignore_index=True)
    if data.empty: raise RuntimeError('No examples loaded')
    assert set(data.gold.unique()) <= {0,1}
    chosen=[]
    for ds,part in data.groupby('dataset'):
        val=sample(part[part.split=='validation'],N_VALIDATION)
        test=sample(part[part.split=='test'],N_TEST)
        assert set(val.group).isdisjoint(set(test.group)), f'Source leakage in {ds}'
        if val.gold.nunique()!=2 or test.gold.nunique()!=2:
            raise RuntimeError(f'{ds} sample needs both labels; increase the sample size or change seed')
        print(ds, 'validation',len(val),'test',len(test), 'positive counts',val.gold.sum(),test.gold.sum())
        chosen.extend([val,test])
    return pd.concat(chosen,ignore_index=True)

def model_for(name, device):
    if name == "AlignScore-base":
        from alignscore import AlignScore
        checkpoint = hf_hub_download(repo_id="yzha/AlignScore", filename="AlignScore-base.ckpt")
        return AlignScore(model="roberta-base", batch_size=1, device=device, ckpt_path=checkpoint, evaluation_mode="nli_sp")
    elif name == "SummaC-Conv":
        from summac.model_summac import SummaCConv
        return SummaCConv(models=["vitc"], bins="percentile", granularity="sentence", nli_labels="e", device="cuda" if device.startswith("cuda") else "cpu", start_file="default", agg="mean")
    elif name == "BERTScore":
        from bert_score import BERTScorer
        return BERTScorer(model_type="roberta-large", lang="en", device=device, batch_size=1, rescale_with_baseline=False)
    elif name == "BARTScore":
        import sys
        if str(BARTSCORE_DIR) not in sys.path:
            sys.path.insert(0, str(BARTSCORE_DIR))
        from bart_score import BARTScorer
        return BARTScorer(device=device, checkpoint="facebook/bart-large-cnn")
    else:
        raise ValueError(f"Unknown method: {name}")

def progress_snapshot(df, counts, phase, current_method='', current_row=None, started=None, baseline=0):
    """Progress is updated for EVERY example; scoring results are saved every 10."""
    n = len(df)
    total = n * len(METHODS)
    completed = sum(counts.values())
    elapsed = time.time() - started if started is not None else 0
    new_scores = completed - baseline
    remaining = max(0, total - completed) * elapsed / new_scores if new_scores > 0 else None
    if completed == total and phase == 'Metrics saved':
        remaining = 0
    datasets = sorted(DATASET_TOTALS)
    benchmarks = {}
    for ds in datasets:
        count = DATASET_TOTALS[ds]
        benchmarks[ds] = {
            'completed': sum(counts.get((method, ds, split), 0) for method in METHODS for split in ('validation','test')),
            'total': count * len(METHODS),
        }
    methods = {}
    for method in METHODS:
        methods[method] = {'completed': sum(counts.get((method,ds,split),0) for ds in datasets for split in ('validation','test')),
                           'total': n}
    ds = str(current_row['dataset']) if current_row is not None else ''
    split = str(current_row['split']) if current_row is not None else ''
    def section(num, key):
        done = counts.get(key, 0) if isinstance(key, tuple) and len(key)==3 else sum(counts.get(k,0) for k in key)
        return {'completed': done, 'total': num, 'percent': 100 * done / num if num else 0}
    m_part = methods.get(current_method, {'completed':0,'total':0})
    m_part['percent'] = 100 * m_part['completed']/m_part['total'] if m_part['total'] else 0
    bench_part = section(DATASET_TOTALS[ds], [(current_method,ds,s) for s in ('validation','test')]) if ds else {}
    task_part = section(TASK_TOTALS[(ds, split)], (current_method,ds,split)) if ds else {}
    return dict(phase=phase, percent=100*completed/total if total else 0,
        completed=completed, total=total, remaining_seconds=remaining,
        current_method=current_method, current_dataset=ds, current_split=split,
        current_id=str(current_row['id']) if current_row is not None else '',
        current_method_progress=m_part, current_benchmark=bench_part, current_task=task_part,
        benchmarks=benchmarks, methods=methods, updated_at=time.time(), elapsed_seconds=elapsed)


def save_progress(df, counts, phase, current_method='', current_row=None, started=None, baseline=0):
    state = progress_snapshot(df, counts, phase, current_method, current_row, started, baseline)
    temp = PROGRESS_FILE.with_suffix('.json.tmp')
    temp.write_text(json.dumps(state, indent=2), encoding='utf-8')
    temp.replace(PROGRESS_FILE)


def read_saved(df, method):
    """Read a completed method CSV, or only its committed checkpoint batches."""
    final = OUT / (method.replace('-','_') + '_scores.csv')
    folder = OUT / 'checkpoints' / method.replace('-', '_')
    if final.exists():
        saved = pd.read_csv(final, dtype={'id':str, 'group':str})
    else:
        batches = sorted(folder.glob('batch_*.csv'))
        saved = pd.concat([pd.read_csv(p, dtype={'id':str, 'group':str}) for p in batches], ignore_index=True) if batches else pd.DataFrame()
    if len(saved):
        if len(saved) > len(df):
            raise RuntimeError(f'Too many saved examples for {method}; use a different results folder')
        expected = list(zip(df.dataset.iloc[:len(saved)], df.id.astype(str).iloc[:len(saved)]))
        actual = list(zip(saved.dataset, saved.id.astype(str)))
        if actual != expected or not (saved.method == method).all():
            raise RuntimeError(f'Checkpoint does not match the selected examples for {method}')
        if not np.isfinite(pd.to_numeric(saved.hallucination_score, errors='coerce')).all():
            raise RuntimeError(f'Invalid scores in {method} checkpoint')
    return saved, folder, final


def score_all(df, name, device, counts, started, baseline):
    saved, folder, path = read_saved(df, name)
    done = len(saved)
    if done == len(df):
        print(f'{name}: using {done} previously saved scores', flush=True)
        if not path.exists():
            saved.to_csv(path, index=False)
        if folder.exists():
            shutil.rmtree(folder)
        return saved
    folder.mkdir(parents=True, exist_ok=True)
    print(f'{name}: {done}/{len(df)} scored, resuming remaining examples', flush=True)
    save_progress(df, counts, 'Loading ' + name, name, df.iloc[done].to_dict(), started, baseline)
    model = model_for(name, device)
    output = [saved] if done else []
    pending = []
    for pos in tqdm(range(done, len(df)), desc=name, initial=done, total=len(df)):
        item = df.iloc[pos].to_dict()
        if device.startswith('cuda'): torch.cuda.synchronize()
        start=time.perf_counter()
        with torch.inference_mode():
            if name == "AlignScore-base":
                score = float(model.score(contexts=[item["context"]], claims=[item["response"]])[0])
                hallucination_score = 1.0 - score
            elif name == "SummaC-Conv":
                score = float(model.score([item["context"]], [item["response"]])["scores"][0])
                hallucination_score = 1.0 - score
            elif name == "BERTScore":
                _, _, f1 = model.score(cands=[item["response"]], refs=[item["context"]], batch_size=1)
                score = float(f1[0].item())
                hallucination_score = 1.0 - score
            elif name == "BARTScore":
                score = float(model.score([item["context"]], [item["response"]], batch_size=1)[0])
                hallucination_score = -score
            else:
                raise ValueError(f"Unknown method: {name}")
        if device.startswith('cuda'): torch.cuda.synchronize()
        seconds = time.perf_counter() - start
        pending.append({**item, 'method': name, 'consistency_score': score,
                        'hallucination_score': hallucination_score, 'seconds': seconds})
        counts[(name,item['dataset'],item['split'])] = counts.get((name,item['dataset'],item['split']), 0) + 1
        save_progress(df, counts, 'Scoring examples', name, item, started, baseline)
        if len(pending) == 10 or pos == len(df) - 1:
            # Atomic batch commit: even a VM crash during writing cannot corrupt prior batches.
            filename = folder / f'batch_{pos-len(pending)+1:08d}.csv'
            temp = filename.with_suffix('.csv.tmp')
            batch = pd.DataFrame(pending)
            batch.to_csv(temp, index=False)
            temp.replace(filename)
            output.append(batch)
            pending = []
    result = pd.concat(output, ignore_index=True)
    result.to_csv(path.with_suffix('.csv.tmp'), index=False)
    path.with_suffix('.csv.tmp').replace(path)
    shutil.rmtree(folder)
    del model
    if device.startswith('cuda'): torch.cuda.empty_cache()
    return result

def assess(all_scores):
    summaries=[]; predictions=[]
    for (method,ds),g in all_scores.groupby(['method','dataset']):
        val=g[g.split=='validation']; test=g[g.split=='test'].copy()
        values=np.sort(val.hallucination_score.unique())
        candidates=np.r_[np.nextafter(values[0],-np.inf),values,np.nextafter(values[-1],np.inf)]
        threshold=float(max(candidates,key=lambda t:f1_score(val.gold,val.hallucination_score>=t,zero_division=0)))
        test['predicted']=(test.hallucination_score>=threshold).astype(int)
        y=test.gold.to_numpy(); pred=test.predicted.to_numpy(); s=test.hallucination_score.to_numpy()
        precision,recall,f1,_=precision_recall_fscore_support(y,pred,average='binary',zero_division=0)
        cm=confusion_matrix(y,pred,labels=[0,1])
        tn,fp,fn,tp=cm.ravel()
        assert tp+fn==int(y.sum()) and tn+fp==int((1-y).sum())
        summaries.append(dict(dataset=ds,method=method,n=len(test),positive_count=int(y.sum()),
            prevalence=float(y.mean()),threshold=threshold,auroc=float(roc_auc_score(y,s)),
            average_precision=float(average_precision_score(y,s)),precision=float(precision),recall=float(recall),
            f1=float(f1),mean_seconds=float(test.seconds.mean()),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp)))
        predictions.append(test)
    return pd.DataFrame(summaries),pd.concat(predictions,ignore_index=True)

def main():
    df = prepare()
    selected = OUT / 'selected_examples.csv'
    if selected.exists():
        previous = pd.read_csv(selected, dtype={'id': str, 'group': str})
        match = len(previous) == len(df) and all(
            previous[column].fillna('').astype(str).tolist() == df[column].fillna('').astype(str).tolist()
            for column in ('dataset','id','group','split','context','response','gold'))
        if not match:
            raise RuntimeError('Selected examples differ from the previous run. Use a new results folder to avoid mixing scores.')
    else:
        temp = selected.with_suffix('.csv.tmp')
        df.to_csv(temp, index=False)
        temp.replace(selected)
    global DATASET_TOTALS, TASK_TOTALS
    DATASET_TOTALS = df.dataset.value_counts().to_dict()
    TASK_TOTALS = df.groupby(['dataset', 'split']).size().to_dict()
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    if device == 'cpu' and os.environ.get('FACTUALITY_REQUIRE_GPU') == '1':
        raise RuntimeError('No CUDA GPU detected on this VM. Check NVIDIA drivers or set REQUIRE_GPU_ON_VM=False.')
    print('Device:', device, flush=True)
    counts = {}
    for method in METHODS:
        saved, _, _ = read_saved(df, method)
        if not saved.empty:
            for (ds, split), group in saved.groupby(['dataset','split']):
                counts[(method,ds,split)] = len(group)
    baseline = sum(counts.values())
    started = time.time()
    save_progress(df, counts, 'Starting model evaluations', started=started, baseline=baseline)
    scores = pd.concat([score_all(df, method, device, counts, started, baseline) for method in METHODS], ignore_index=True)
    save_progress(df, counts, 'Calculating validation thresholds and test metrics', started=started, baseline=baseline)
    summary, preds = assess(scores)
    summary.to_csv(OUT/'metrics.csv',index=False)
    preds.to_csv(OUT/'test_predictions.csv',index=False)
    preds[preds.gold!=preds.predicted].to_csv(OUT/'test_errors.csv',index=False)
    metadata={'seed':SEED,'validation_limit':N_VALIDATION,'test_limit':N_TEST,'device':device,
      'threshold':'per dataset and method, maximizing validation F1',
      'score_direction':'negative consistency score; higher means more likely hallucination',
      'label_policies':{'RAGTruth':'any annotated span, including implicit_true',
       'HaluEval':'right_summary=0; hallucinated_summary=1 (dataset-designated)',
       'FaithBench':'worst severity: Questionable or Unwanted=1; Benign-only or no annotations=0'},
      'dataset_commits':{name:subprocess.check_output(['git','-C',str(ROOT/name),'rev-parse','HEAD'],text=True).strip() for name in ['RAGTruth','HaluEval','FaithBench']}}
    (OUT/'settings.json').write_text(json.dumps(metadata,indent=2))
    save_progress(df, counts, 'Metrics saved', started=started, baseline=baseline)
    print('\nTEST RESULTS\n', summary.to_string(index=False)); print('\nSaved to', OUT)

if __name__ == '__main__':
    main()


In [ ]:
with report_errors("Evaluating the full summarization corpus"):
    import os, subprocess

    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    log_path = RUN_DIR / "execution.log"
    process = subprocess.Popen(
        [PYTHON, "-u", str(EVAL_SCRIPT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    REPORTER.evaluator_pid = process.pid
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if process.wait() != 0:
        error_lines = log_path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()[-35:]
        raise RuntimeError("Evaluation failed. Last output:\n" + "\n".join(error_lines))

## 4. View and download results
Saved outputs include the full selected pool, model score CSVs, metrics, predictions, errors, exact dataset commits and environment versions. The results ZIP is stored next to the run folder on a VM or on Google Drive in Colab. The completion email attempts to attach the ZIP; if it is too large or SMTP rejects it, a notification is sent without the attachment. Colab also keeps its original automatic browser ZIP download.


In [ ]:
with report_errors("Packing and emailing final results"):
    import pandas as pd, shutil
    from IPython.display import display

    RESULTS = RUN_DIR
    display(pd.read_csv(RESULTS / "metrics.csv"))
    display(pd.read_csv(RESULTS / "test_errors.csv").head())
    with open(RESULTS / "environment.txt", "w") as f:
        subprocess.run(
            ["uv", "pip", "freeze", "--python", PYTHON], stdout=f, check=True
        )
    shutil.copy(EVAL_SCRIPT, RESULTS / "evaluate_comparison.py")
    if LIVE_PROGRESS_FILE != RESULTS / "progress.json":
        shutil.copy(LIVE_PROGRESS_FILE, RESULTS / "progress.json")
    zip_path = shutil.make_archive(
        str(RUN_DIR.parent / RUN_DIR.name), "zip", root_dir=RESULTS
    )
    print("Results folder:", RUN_DIR)
    print("ZIP saved at:", zip_path)
    REPORTER.stop()
    REPORTER.send("COMPLETED", attachment=zip_path)
    if IN_COLAB:
        from google.colab import files

        files.download(zip_path)